In [2]:
from transformers import AutoTokenizer, AutoModel, AutoConfig
import torch
import pandas as pd
import pickle
from tqdm import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
df = pd.read_csv("data/prot_id_content.csv")
df


,id,content
0,901,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...
1,902,MALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPF...
2,903,MAALRQPQVAELLAEARRAFREEFGAEPELAVSAPGRVNLIGEHTD...
3,904,MENTENSVDSKSIKNLEPKIIHGSESMDSGISLDNSYKMDYPEMGL...
4,905,MADKVLKEKRKLFIRSMGEGTINGLLDELLQTRVLNKEEMEKVKRE...
...,...,...
61009,1174163,MVLTKTATNDESVCTMFGSRYVRTTLPKYEIGENSIPKDAAYQIIK...
61010,1174164,MDANVVSSSTIATYIDALAKNASELEQRSTAYEINNELELVFIKPP...
61011,1174166,MKKNTDSEMDQRLGYKFLVPDPKAGVFYRPLHFQYVSYSNFILHRL...
61012,1174167,MPQQLSPINIETKKAISNARLKPLDIHYNESKPTTIQNTGKLVRIN...


In [5]:
len(df["content"])

61014

In [6]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D", do_lower_case=False)
model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(device)
model.eval()
emb_dict = {}
for i in tqdm(range(len(df["content"]))):
    encoded_input = tokenizer(df["content"][i], return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    protein_embedding = output.last_hidden_state
    protein_embedding = protein_embedding.squeeze()
    protein_embedding = protein_embedding.mean(dim=0)
    emb_dict[df['id'][i].item()] = protein_embedding.half().cpu()       # x: [D] или [1,D]


Loading weights: 100%|██████████| 209/209 [00:00<00:00, 1710.73it/s, Materializing param=encoder.layer.11.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 61014/61014 [17:02<00:00, 59.66it/s]  


In [7]:
with open("data/dict.pkl", "wb") as f:
    pickle.dump(emb_dict, f)

In [11]:
# Load model directly


tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True)
model = AutoModel.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True).to(device)

dna = "CUGCUGCUGCUGCUGCUG"
inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
output = model(**encoded_input)

# Эмбеддинги (последний слой)
dna_embeddings = output.last_hidden_state
dna_embeddings = dna_embeddings.squeeze()
dna_embeddings

RuntimeError: Tensor on device meta is not on the expected device cpu!

In [88]:
tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-100M-MLM")
model = AutoModel.from_pretrained("DeepChem/ChemBERTa-100M-MLM").to(device)
model.eval()
buf_sm = []
for i in df_sm["n.content"]:
    encoded_input = tokenizer(i, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    sm_embedding = output.last_hidden_state
    sm_embedding = sm_embedding.squeeze()
    sm_embedding = sm_embedding.mean(dim=0)
    buf_sm.append(sm_embedding.half().cpu())        # x: [D] или [1,D]

T_sm = torch.stack(buf_sm, dim=0)  # итоговый [N,D]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2557.19it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
tokenizer = AutoTokenizer.from_pretrained("PharMolix/BioMedGPT-R1", trust_remote_code=True)
model = AutoModel.from_pretrained("PharMolix/BioMedGPT-R1").to(device)
model.eval()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/PharMolix/BioMedGPT-R1.
401 Client Error. (Request ID: Root=1-698b41ed-5740fb9672481596292eff61;ba2c403e-895c-4052-8a80-a24334e35bd0)

Cannot access gated repo for url https://huggingface.co/PharMolix/BioMedGPT-R1/resolve/main/config.json.
Access to model PharMolix/BioMedGPT-R1 is restricted. You must have access to it and be authenticated to access it. Please log in.